# Assignment 2.1: Naive Bayes on text

The Naive Bayes variants are already implemented in this notebook. Your job is to decide which
variant belongs on which dataset, call it with the right arguments, run the experiments, and
write the report.

Three code cells are yours: `run_multinomial`, `run_bernoulli`, and the experiment table.
Everything else is provided and already runs.

# Import libraries
Do not use any other Python library.

numpy - Linear algebra library for handling vectors and matrices, collectively processed as numpy arrays.

matplotlib - Graphing library for visualizing results.

csv - Library used for reading csv files into python lists.

sklearn - Machine learning library from which we source two datasets, the word tokenizer, the confusion matrix plot, and the reference implementations of the two Naive Bayes variants you compare against.

time - Simple library for timing code.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from csv import reader
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from time import time

# Load Datasets

These functions load the two text datasets and split every document into lowercase word tokens.
The tokenizer is the one sklearn's CountVectorizer uses; nothing else from that class is used
here. The 20 Newsgroups loader drops the headers, the footers and the quoted replies, so the
models learn from the body text alone. It downloads about 14 MB the first time it runs and caches
it under `~/scikit_learn_data`.

Outputs:

*   **documents**: list of N documents, each one a list of word tokens
*   **labels**: list of N integer class labels
*   **label_names**: list of class names, indexed by the label

In [ ]:
def load_spam_ham():
    # kaggle spam email dataset, one email per row
    with open('../data/classification-datasets/emails.csv', 'r') as file:
        rows = list(reader(file))[1:]
    tokenizer = CountVectorizer().build_tokenizer()
    # every email starts with "Subject: ", which is the same in both classes
    documents = [tokenizer(row[0][9:].lower()) for row in rows]
    labels = [int(row[1]) for row in rows]
    return documents, labels, ['ham', 'spam']


def load_newsgroups():
    # headers, footers and quoted replies give the topic away, so they come out
    newsgroups = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'))
    tokenizer = CountVectorizer().build_tokenizer()
    documents = [tokenizer(text.lower()) for text in newsgroups.data]
    labels = [int(label) for label in newsgroups.target]
    return documents, labels, list(newsgroups.target_names)

# Function: train_prior

Estimates p(y = l) as the fraction of the training labels that carry class l. That is the prior
term in Equations 1 and 2 of the spec.

Inputs:
*   **labels**: list of training labels

Output:
*   **class_priors**: dictionary mapping each class to its prior probability (float)

In [ ]:
def train_prior(labels):
    class_counts = {}
    for label in labels:
        class_counts[label] = class_counts.get(label, 0) + 1
    return {label: count / len(labels) for label, count in class_counts.items()}

# Function: train_multinomial

Collects the counts the Multinomial estimate p_M(w | y = l) needs, Equation 4 of the spec: how
many times each word occurs in each class, and how many words each class holds in total. The
vocabulary V is built from the training documents alone.
Here k is the Laplace smoothing constant, which adds k to every count so that no probability is
ever zero. This project uses k = 1.

Inputs:
*   **documents**: list of documents, each one a list of word tokens
*   **labels**: list of labels, one per document
*   **smoothing**: Laplace smoothing constant k (float)

Output:
*   **model**: dictionary holding the word counts, the per class word totals, the vocabulary, and the smoothing constant

# Function: predict_multinomial

Scores every document under every class with Equation 2, using the Multinomial estimate
p_M(w | y = l) from Equation 4. A word that occurs twice contributes two terms. A word of the
vocabulary that the document does not contain contributes nothing here, unlike Bernoulli. A word
that is out of vocabulary is skipped. The log probability of each word under each class is computed
once up front rather than once per document.

Inputs:
*   **documents**: list of documents, each one a list of word tokens
*   **model**: the dictionary returned by train_multinomial
*   **class_priors**: dictionary mapping each class to its prior probability (float)

Output:
*   **log_probabilities**: list with one dictionary per document, mapping each class to its log probability

In [ ]:
def train_multinomial(documents, labels, smoothing=1):
    word_counts = {}
    total_words = {}
    vocabulary = set()
    for document, label in zip(documents, labels):
        counts = word_counts.setdefault(label, {})
        total_words[label] = total_words.get(label, 0) + len(document)
        for word in document:
            counts[word] = counts.get(word, 0) + 1
            vocabulary.add(word)
    return {'word_counts': word_counts, 'total_words': total_words,
            'vocabulary': vocabulary, 'smoothing': smoothing}


def predict_multinomial(documents, model, class_priors):
    vocabulary = model['vocabulary']
    smoothing = model['smoothing']
    vocabulary_size = len(vocabulary)

    # one log probability table per class, built once
    log_word_probabilities = {}
    unseen_log_probabilities = {}
    for label, counts in model['word_counts'].items():
        denominator = model['total_words'][label] + smoothing * vocabulary_size
        log_word_probabilities[label] = {
            word: np.log((count + smoothing) / denominator) for word, count in counts.items()}
        # a vocabulary word this class never used still gets the smoothed numerator
        unseen_log_probabilities[label] = np.log(smoothing / denominator)
    log_priors = {label: np.log(prior) for label, prior in class_priors.items()}

    log_probabilities = []
    for document in documents:
        # out of vocabulary words are dropped here
        counts = {}
        for word in document:
            if word in vocabulary:
                counts[word] = counts.get(word, 0) + 1
        class_log_probabilities = {}
        for label, word_probabilities in log_word_probabilities.items():
            unseen = unseen_log_probabilities[label]
            total = log_priors[label]
            for word, count in counts.items():
                total += count * word_probabilities.get(word, unseen)
            class_log_probabilities[label] = total
        log_probabilities.append(class_log_probabilities)
    return log_probabilities

# Function: train_bernoulli

Collects the counts the Multi-Variate Bernoulli estimate p_B(x_w | y = l) needs, Equation 3 of the
spec: how many documents of each class contain each word at least once, and how many documents
each class holds. A repeated word counts once.

Inputs:
*   **documents**: list of documents, each one a list of word tokens
*   **labels**: list of labels, one per document
*   **smoothing**: Laplace smoothing constant k (float)

Output:
*   **model**: dictionary holding the per class document counts, the per word document counts, the vocabulary, and the smoothing constant

# Function: predict_bernoulli

Scores every document under every class with the adaptation of Equation 2 that sums over every
word of the vocabulary, using the Multi-Variate Bernoulli estimate p_B(x_w | y = l) from Equation
3. A word the document does not contain still contributes a term, ln(1 - p). That absence is
evidence in its own right: a word that is common in one class but missing from the document pushes
the score away from that class. This is what separates Bernoulli from Multinomial, which only
counts the words that are present. A word that is out of vocabulary is skipped. Summing over the whole vocabulary for every
document would be far too slow, so the function sums ln(1 - p) over the vocabulary once per class
and then corrects that sum for the words the document does contain.

Inputs:
*   **documents**: list of documents, each one a list of word tokens
*   **model**: the dictionary returned by train_bernoulli
*   **class_priors**: dictionary mapping each class to its prior probability (float)

Output:
*   **log_probabilities**: list with one dictionary per document, mapping each class to its log probability

In [ ]:
def train_bernoulli(documents, labels, smoothing=1):
    document_counts = {}
    word_document_counts = {}
    vocabulary = set()
    for document, label in zip(documents, labels):
        document_counts[label] = document_counts.get(label, 0) + 1
        counts = word_document_counts.setdefault(label, {})
        for word in set(document):
            counts[word] = counts.get(word, 0) + 1
            vocabulary.add(word)
    return {'document_counts': document_counts, 'word_document_counts': word_document_counts,
            'vocabulary': vocabulary, 'smoothing': smoothing}


def predict_bernoulli(documents, model, class_priors):
    vocabulary = model['vocabulary']
    smoothing = model['smoothing']

    absent_totals = {}
    present_corrections = {}
    absent_corrections = {}
    for label, counts in model['word_document_counts'].items():
        denominator = model['document_counts'][label] + 2 * smoothing
        unused_probability = smoothing / denominator
        # every word this class never used shares the same ln(1 - p) term
        total = (len(vocabulary) - len(counts)) * np.log(1.0 - unused_probability)
        corrections = {}
        for word, count in counts.items():
            probability = (count + smoothing) / denominator
            total += np.log(1.0 - probability)
            # swapping ln(1 - p) for ln(p) when the word turns up
            corrections[word] = np.log(probability) - np.log(1.0 - probability)
        absent_totals[label] = total
        present_corrections[label] = corrections
        absent_corrections[label] = np.log(unused_probability) - np.log(1.0 - unused_probability)
    log_priors = {label: np.log(prior) for label, prior in class_priors.items()}

    log_probabilities = []
    for document in documents:
        # out of vocabulary words are dropped here
        present = set(document) & vocabulary
        class_log_probabilities = {}
        for label, corrections in present_corrections.items():
            absent = absent_corrections[label]
            total = log_priors[label] + absent_totals[label]
            for word in present:
                total += corrections.get(word, absent)
            class_log_probabilities[label] = total
        log_probabilities.append(class_log_probabilities)
    return log_probabilities

# Function: predicted_labels

Turns per class log probabilities into one label per instance, the arg max of Equation 2.

Input:
*   **log_probabilities**: list with one dictionary per instance, mapping each class to its log probability

Output:
*   **predicted**: list of predicted labels, one per instance

# Function: accuracy

Percentage of instances whose predicted label matches the true label.

Inputs:
*   **true_labels**: list of true labels
*   **predicted**: list of predicted labels

Output:
*   **accuracy**: percentage of correct predictions (float)

# Function: show_confusion_matrix

Draws one confusion matrix. Rows are true classes, columns are predicted classes.

Inputs:
*   **true_labels**: list of true labels
*   **predicted**: list of predicted labels
*   **label_names**: list of class names, indexed by the label
*   **title**: title for the figure (string)

In [ ]:
def predicted_labels(log_probabilities):
    return [max(row, key=row.get) for row in log_probabilities]


def accuracy(true_labels, predicted):
    correct = sum(1 for true, prediction in zip(true_labels, predicted) if true == prediction)
    return correct / len(true_labels) * 100


def show_confusion_matrix(true_labels, predicted, label_names, title):
    matrix = confusion_matrix(true_labels, predicted)
    display = ConfusionMatrixDisplay(confusion_matrix=matrix, display_labels=label_names)
    # twenty newsgroup classes need a bigger figure than two spam classes
    size = 5 if len(label_names) <= 4 else 11
    figure, axes = plt.subplots(figsize=(size, size))
    display.plot(cmap=plt.cm.Blues, ax=axes, colorbar=False, values_format='d',
                 xticks_rotation='vertical')
    axes.set_title(title)
    plt.show()

# Function: fold_indices

Shuffles the instance indices once with seed 42 and cuts them into k folds. Each fold is the test
set exactly once, so every instance is tested exactly once across the k runs.

Inputs:
*   **n_samples**: number of instances (integer)
*   **k_folds**: number of folds (integer)
*   **seed**: seed for the shuffle (integer)

Output:
*   **splits**: list of (train_indices, test_indices) pairs, one per fold

# Function: cross_validate

Runs one experiment function over the k folds and collects the results. The experiment function is
called once per fold as `run_experiment(*train_tables, train_labels, *test_tables)` and returns one
predicted label per test instance.

Inputs:
*   **tables**: list of feature tables, each one N rows long and in the same row order
*   **labels**: list of N labels
*   **run_experiment**: function that trains on a fold and returns predicted labels
*   **k_folds**: number of folds (integer)

Outputs:
*   **fold_accuracies**: list of the accuracy on each held out fold (floats)
*   **true_labels**: true labels of every instance, in the order they were tested
*   **predicted**: predicted labels of every instance, in the same order

In [ ]:
def fold_indices(n_samples, k_folds=5, seed=42):
    order = np.random.default_rng(seed).permutation(n_samples)
    folds = np.array_split(order, k_folds)
    splits = []
    for held_out in range(k_folds):
        train_indices = [int(i) for fold in range(k_folds) if fold != held_out
                         for i in folds[fold]]
        test_indices = [int(i) for i in folds[held_out]]
        splits.append((train_indices, test_indices))
    return splits


def cross_validate(tables, labels, run_experiment, k_folds=5):
    fold_accuracies = []
    true_labels = []
    predicted = []
    for train_indices, test_indices in fold_indices(len(labels), k_folds):
        train_tables = [[table[i] for i in train_indices] for table in tables]
        test_tables = [[table[i] for i in test_indices] for table in tables]
        train_labels = [labels[i] for i in train_indices]
        fold_true = [labels[i] for i in test_indices]
        fold_predicted = run_experiment(*train_tables, train_labels, *test_tables)
        fold_accuracies.append(accuracy(fold_true, fold_predicted))
        true_labels.extend(fold_true)
        predicted.extend(fold_predicted)
    return fold_accuracies, true_labels, predicted

# Function: run_library_text

Wraps a scikit-learn classifier so it runs one fold with the same signature as `run_multinomial`,
which lets the cross validation harness score it on exactly the same folds. `CountVectorizer` is
told to take each document as it already is, because the loaders return token lists rather than
raw strings. It learns the vocabulary from the training documents alone and drops test words that
are not in it, the same way the provided code does. `BernoulliNB` turns the counts into present or
absent before it trains, which is what makes it the Multi-Variate Bernoulli variant. `alpha` is
scikit-learn's name for the Laplace smoothing constant k.

Inputs:
*   **model**: a scikit-learn classifier, such as MultinomialNB(alpha=1) or BernoulliNB(alpha=1)

Output:
*   **run_experiment**: function with the same signature as run_multinomial, returning one predicted label per test document

In [ ]:
def run_library_text(model):
    def run_experiment(train_documents, train_labels, test_documents):
        # the documents are already tokenized, so the vectorizer takes each list as it is
        vectorizer = CountVectorizer(analyzer=lambda document: document)
        # the vocabulary comes from the training documents, test words outside it are dropped
        train_matrix = vectorizer.fit_transform(train_documents)
        test_matrix = vectorizer.transform(test_documents)
        model.fit(train_matrix, train_labels)
        return [int(label) for label in model.predict(test_matrix)]
    return run_experiment

# Function: run_multinomial

Runs the Multinomial Naive Bayes experiment on one fold. The cross validation harness calls this
once per fold with that fold's training documents and labels and the held out test documents, and
expects one predicted label per test document. Use smoothing k = 1.

Inputs:
*   **train_documents**: list of training documents, each one a list of word tokens
*   **train_labels**: list of training labels
*   **test_documents**: list of held out documents, each one a list of word tokens

Output:
*   **predicted**: list of predicted labels, one per test document

In [ ]:
def run_multinomial(train_documents, train_labels, test_documents):
    # 1. estimate the class priors from the training labels
    # 2. train the variant that counts how many times each word occurs, with smoothing k = 1
    # 3. score the held out documents with the predict function that matches that variant
    # 4. return one label per test document
    # TODO
    pass

# Function: run_bernoulli

Runs the Multi-Variate Bernoulli Naive Bayes experiment on one fold. Same inputs and output as
`run_multinomial`. Use smoothing k = 1.

Inputs:
*   **train_documents**: list of training documents, each one a list of word tokens
*   **train_labels**: list of training labels
*   **test_documents**: list of held out documents, each one a list of word tokens

Output:
*   **predicted**: list of predicted labels, one per test document

In [ ]:
def run_bernoulli(train_documents, train_labels, test_documents):
    # 1. estimate the class priors from the training labels
    # 2. train the variant that counts whether each word is present or absent, with smoothing k = 1
    # 3. score the held out documents with the predict function that matches that variant
    # 4. return one label per test document
    # TODO
    pass

# Experiment table

List the runs you want. Each entry gives a short name for the variant, the function that runs one
fold of it, and the datasets it applies to. The dataset names are the keys of the `datasets`
dictionary in the driver cell below: `spam-ham` and `newsgroups`. The spec says which variants
belong on which datasets.

Each variant also gets a scikit-learn run beside it. `run_library_text(model)` turns a scikit-learn
classifier into a run function of the same shape, so a library entry goes in the table exactly like
your own.

In [ ]:
# 1. one entry per variant, keyed by the short name that will label the tables and figures
# 2. one more entry per matching scikit-learn class, so both run on the same folds
# 3. each entry names the function that runs one fold and lists the datasets that variant applies to
# 4. every entry uses smoothing k = 1
# TODO
experiments = {}

# Main Naive Bayes code

This cell runs the experiments. For each dataset it loads the documents, runs every variant your
table lists for that dataset under 5 fold cross validation, prints the accuracy on each fold and
the mean over the folds, and draws one confusion matrix per dataset and variant. Each confusion
matrix covers all five test folds, so every document is counted exactly once.

Newsgroups takes about a minute for both variants together, plus a one time download.

In [ ]:
datasets = {
    "spam-ham": load_spam_ham,
    "newsgroups": load_newsgroups,
}

k_folds = 5
results = {}

for dataset_name, load_dataset in datasets.items():
    documents, labels, label_names = load_dataset()
    print(f"{dataset_name}: {len(documents)} documents, {len(label_names)} classes")

    for variant_name, variant in experiments.items():
        if dataset_name not in variant["datasets"]:
            continue
        print(f"RUNNING {dataset_name}, {variant_name}")
        start_time = time()
        fold_accuracies, true_labels, predicted = cross_validate(
            [documents], labels, variant["run"], k_folds)
        elapsed_time = time() - start_time

        results[(dataset_name, variant_name)] = np.mean(fold_accuracies)
        print("fold accuracies: " + ", ".join(f"{value:.2f}%" for value in fold_accuracies))
        print(f"mean accuracy: {np.mean(fold_accuracies):.2f}%")
        print(f"elapsed time: {elapsed_time:.1f}s")
        show_confusion_matrix(true_labels, predicted, label_names,
                              f"{dataset_name}, {variant_name}")

print("mean accuracy over 5 folds")
for (dataset_name, variant_name), value in results.items():
    print(f"  {dataset_name:12s} {variant_name:20s} {value:.2f}%")

print("provided code against scikit-learn")
for dataset_name in datasets:
    for variant_name in ("multinomial", "bernoulli"):
        provided = results.get((dataset_name, variant_name))
        library = results.get((dataset_name, f"sklearn {variant_name}"))
        # a pair the experiment table left out has nothing to compare
        if provided is None or library is None:
            continue
        print(f"  {dataset_name:12s} {variant_name:12s} provided {provided:.2f}%, "
              f"scikit-learn {library:.2f}%, difference {provided - library:+.2f} points")

# Report

Cover the following for this notebook.

*   A table of the mean accuracy over the five folds, one row per dataset and variant.
*   A row in that table for the scikit-learn run of each dataset and variant, beside the provided one.
*   The confusion matrix for each dataset and variant.
*   Multinomial against Multi-Variate Bernoulli on each dataset: which one wins, by how much, and why.
*   Provided code against scikit-learn on each dataset: are the accuracies the same? If they differ, say what in the two implementations could explain it (tokenization, smoothing, how the vocabulary is built, how absent words are treated).
*   The classes 20 Newsgroups confuses most, read off the confusion matrix, and what those classes share.
*   Why each variant suits text data, in terms of what it assumes about how a document is written.
*   What you would expect to see if you ran the other variant's training function with this one's prediction function.